# Clinical Note Analysis & Synthetic Data Reporting

This notebook covers:
1.  **LLM Setup**: Loading Mistral API or Local Models.
2.  **Dataset Reporting**: Analyzing the synthetic medical dataset found in `./synthetic-medical-dataset`.
3.  **Note Generation**: Creating synthetic clinical notes from structured CSV data (Conditions, Medications, etc.) using the LLM.
4.  **Note Analysis**: Summarizing and extracting entities from these generated notes.

In [13]:
import os
import pandas as pd

# Common Configuration
DATASET_DIR = "./synthetic-medical-dataset"

CLINICAL_NOTE_EXAMPLE = """
Patient: 45-year-old male
Chief Complaint: Severe chest pain radiating to the left arm, shortness of breath.
History: Hypertension, Type 2 Diabetes (diagnosed 2018), Smoker (1 pack/day for 20 years).
Vitals: BP 160/95, HR 105, O2 94%.
Assessment: Suspected Acute Coronary Syndrome (ACS). ECG shows ST elevation in leads V2-V4.
Plan: Administer Aspirin 325mg, Nitroglycerin sublingual. Urgent Cardiology consult.
"""

## 1. LLM Model Setup

### Option A: Mistral API (Recommended)
**Prerequisites:** `pip install mistralai`

In [14]:
from mistralai import Mistral

# Setup API Client
MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY") or "Gf1eQBxuLh2OWxMLSH7aIAUjpcXF5CfM"

if MISTRAL_API_KEY == "YOUR_MISTRAL_API_KEY_HERE":
    print("⚠️ WARNING: Please replace 'YOUR_MISTRAL_API_KEY_HERE' with your actual API key.")
    client = None
else:
    print("Mistral API Key detected.")
    client = Mistral(api_key=MISTRAL_API_KEY)

def analyze_with_mistral(note_text, task="summarize"):
    if not client:
        return "Mistral Client not initialized (check API Key)."
        
    if task == "summarize":
        content = f"Please provide a concise medical summary of the following clinical note:\n\n{note_text}"
    elif task == "extract_entities":
        content = f"Extract the key medical entities (Diseases, Medications, Symptoms, Vitals) from this note as a JSON list:\n\n{note_text}"
    elif task == "generate_note":
        content = f"Generate a realistic clinical note based on this structured patient data. Include Chief Complaint, HPI, Assessment, and Plan:\n\n{note_text}"
    else:
        content = note_text

    try:
        chat_response = client.chat.complete(
            model="mistral-large-latest",
            messages=[{"role": "user", "content": content}]
        )
        return chat_response.choices[0].message.content
    except Exception as e:
        return f"Error calling Mistral API: {e}"

Mistral API Key detected.


### Option B: Local Hugging Face Model (Offline)

In [ ]:
from transformers import pipeline
import torch

def setup_local_model():
    local_model_name = "distilgpt2" 
    device = 0 if torch.cuda.is_available() else -1
    print(f"Loading local model '{local_model_name}' on device {'GPU' if device==0 else 'CPU'}...")
    return pipeline('text-generation', model=local_model_name, device=device)

# Uncomment to load local model if needed
# local_generator = setup_local_model()

## 2. Dataset Reporting & File Analysis
We will list the files in the dataset directory and analyze their contents.

In [15]:
def analyze_directory(directory):
    if not os.path.exists(directory):
        print(f"Directory not found: {directory}")
        return

    files = os.listdir(directory)
    print(f"Found {len(files)} files in {directory}:")
    
    csv_files = [f for f in files if f.endswith('.csv')]
    
    for f in csv_files:
        file_path = os.path.join(directory, f)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        
        # Read first few lines to check shape
        try:
            df = pd.read_csv(file_path, nrows=5)
            cols = list(df.columns)
            print(f" - {f:<20} | Size: {size_mb:.2f} MB | Columns: {len(cols)} | Example: {cols[:3]}...")
        except Exception as e:
            print(f" - {f:<20} | Error reading: {e}")

analyze_directory(DATASET_DIR)

Found 11 files in ./synthetic-medical-dataset:
 - allergies.csv        | Size: 0.06 MB | Columns: 6 | Example: ['START', 'STOP', 'PATIENT']...
 - all_prevalences.csv  | Size: 0.01 MB | Columns: 6 | Example: ['ITEM', 'POPULATION TYPE', 'OCCURRENCES']...
 - careplans.csv        | Size: 2.21 MB | Columns: 9 | Example: ['ID', 'START', 'STOP']...
 - claims.csv           | Size: 2.84 MB | Columns: 7 | Example: ['ID', 'PATIENT', 'BILLABLEPERIOD']...
 - conditions.csv       | Size: 0.85 MB | Columns: 6 | Example: ['START', 'STOP', 'PATIENT']...
 - encounters.csv       | Size: 2.51 MB | Columns: 7 | Example: ['ID', 'DATE', 'PATIENT']...
 - immunizations.csv    | Size: 1.57 MB | Columns: 5 | Example: ['DATE', 'PATIENT', 'ENCOUNTER']...
 - medications.csv      | Size: 0.91 MB | Columns: 8 | Example: ['START', 'STOP', 'PATIENT']...
 - observations.csv     | Size: 9.26 MB | Columns: 7 | Example: ['DATE', 'PATIENT', 'ENCOUNTER']...
 - patients.csv         | Size: 0.25 MB | Columns: 17 | Example: ['p

### Detailed Data Analysis: Patients & Conditions
Let's load the full `patients.csv` and `conditions.csv` to understand our population.

In [16]:
try:
    patients_df = pd.read_csv(os.path.join(DATASET_DIR, "patients.csv"))
    conditions_df = pd.read_csv(os.path.join(DATASET_DIR, "conditions.csv"))
    
    print("\n--- Patient Demographics ---")
    print(f"Total Patients: {len(patients_df)}")
    print("Gender Distribution:")
    # Corrected column name 'gender' (lowercase)
    print(patients_df['gender'].value_counts())
    
    print("\n--- Top 5 Diagnosed Conditions ---")
    print(conditions_df['DESCRIPTION'].value_counts().head(5))
    
except FileNotFoundError:
    print("Required CSV files not found for detailed analysis.")


--- Patient Demographics ---
Total Patients: 1462
Gender Distribution:
gender
M    741
F    721
Name: count, dtype: int64

--- Top 5 Diagnosed Conditions ---
DESCRIPTION
Viral sinusitis (disorder)            1125
Acute viral pharyngitis (disorder)     602
Acute bronchitis (disorder)            508
Prediabetes                            458
Hypertension                           373
Name: count, dtype: int64


## 3. Synthetic Note Generation
Since the dataset contains structured data (conditions, encounters) but likely no raw free-text notes, we can use our LLM (Mistral) to **generate** realistic clinical notes for a random patient. This is useful for training downstream NLP models.

In [17]:
# Select a random patient
if 'patients_df' in locals() and 'conditions_df' in locals():
    sample_patient = patients_df.sample(1).iloc[0]
    # Corrected column name 'patient' (lowercase) for ID
    pat_id = sample_patient['patient']
    
    # Get their conditions
    pat_conditions = conditions_df[conditions_df['PATIENT'] == pat_id]['DESCRIPTION'].tolist()
    
    # Corrected column names for demographics
    structured_data = f"""
    Patient info: {sample_patient.get('race', 'Unknown')} {sample_patient.get('gender', 'Unknown')}, born {sample_patient.get('birthdate', 'Unknown')}.
    Known Conditions: {', '.join(pat_conditions) if pat_conditions else 'None'}.
    encounter_reason: Routine checkup / Follow-up.
    """
    
    print(f"--- Generating Clinical Note for Patient {pat_id} ---")
    print(f"Input Data: {structured_data.strip()}")
    
    # Generate Note
    if client:
        generated_note = analyze_with_mistral(structured_data, task="generate_note")
        print("\n--- Generated Note ---")
        print(generated_note)
    else:
        print("Skipping generation (Mistral API Key missing).")
else:
    print("Dataframes not loaded.")

--- Generating Clinical Note for Patient 3f4eca54-1246-4102-bc36-c1cf53fe3dbb ---
Input Data: Patient info: white F, born 1973-08-20.
    Known Conditions: Appendicitis, History of appendectomy, Hypertension, Prediabetes, Viral sinusitis (disorder), Viral sinusitis (disorder), Acute viral pharyngitis (disorder), Normal pregnancy, Viral sinusitis (disorder).
    encounter_reason: Routine checkup / Follow-up.

--- Generated Note ---
**Clinical Note**

**Patient Name:** [Patient’s Full Name]
**Date of Birth:** 08/20/1973 (Age: 50)
**Sex:** Female
**Date of Encounter:** [Insert Date]
**Provider:** [Your Name/Title]

---

### **Chief Complaint (CC):**
"Routine follow-up for hypertension, prediabetes, and general wellness check."

---

### **History of Present Illness (HPI):**
The patient is a 50-year-old white female presenting for a routine follow-up visit to monitor her chronic conditions, including hypertension and prediabetes. She reports no acute complaints at this time.

- **Hypertens

## 4. Note Analysis (Entity Extraction)
Now we analyze the note we just generated (or the example note).

In [18]:
target_note = locals().get('generated_note', CLINICAL_NOTE_EXAMPLE)

print("--- Extracting Entities from Target Note ---")
if client:
    entities = analyze_with_mistral(target_note, task="extract_entities")
    print(entities)
else:
    print("Skipping extraction (Mistral API Key missing).")

--- Extracting Entities from Target Note ---
Here is the extracted JSON list of key medical entities from the clinical note:

```json
{
  "medical_entities": {
    "diseases": [
      "hypertension",
      "prediabetes",
      "viral upper respiratory infection",
      "acute appendicitis",
      "type 2 diabetes (family history)",
      "bacterial sinusitis (ruled out)",
      "pharyngitis (ruled out)",
      "postnasal drip (residual)"
    ],
    "medications": [
      "lisinopril 10 mg daily",
      "metformin 500 mg twice daily"
    ],
    "symptoms": [
      "nasal congestion",
      "sore throat",
      "fatigue",
      "polyuria (denied)",
      "polydipsia (denied)",
      "blurred vision (denied)",
      "headaches (denied)",
      "dizziness (denied)",
      "chest pain (denied)",
      "shortness of breath (denied)",
      "vision changes (denied)",
      "weight changes (denied)",
      "fever (denied)",
      "chills (denied)",
      "ear pain (denied)",
      "persistent 